In [1]:
#This code computes medians of errors and uses bootstrapping to calculate errors of medians

In [2]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
import seaborn as sns
import matplotlib.gridspec as gridspec
import ast
import sys
sys.path.append('../no_degeneracy/')
sys.path.append('../no_degeneracy/Prior/')
from mcmc import *
from parallel import *
from fit_prior import read_prior_par
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_absolute_error
from scipy.stats import bootstrap
sns.set_theme(context='paper', style='white', font="Helvetica", font_scale=1, color_codes=True, rc=None)
sns.set_style("ticks")

In [9]:
#Get and sort errors of interpolation
def get_errors(resolution):
    #read errors
    errors_inter_extrapolation=pd.read_csv('../../data/'+ 'errors_interpolation_' + resolution +  '.csv',index_col=0)
    display(errors_inter_extrapolation)
    columns=["sigma", "function","rmse_nn_interp.", "rmse_nn_extrap.", "rmse_mdl_interp.", "rmse_mdl_extrap.", "n", "r"]
    #columns=["sigma", "function","nn interp.", "nn extrap.", "mdl interp.", "mdl extrap.", "n", "r"]
    errors_inter_extrapolation=errors_inter_extrapolation[columns]

    #divide between train/test
    errors_interpolation=errors_inter_extrapolation[["sigma","function", "rmse_nn_interp.", "rmse_mdl_interp.","n", "r"]]
    #errors_interpolation=errors_inter_extrapolation[["sigma","function", "nn interp.", "mdl interp.","n", "r"]]
    errors_interpolation=pd.melt(errors_interpolation,id_vars=["sigma","function", "n", "r"], var_name="error_interp.",
                                 value_name= "value_interp.")   

    errors_extrapolation=errors_inter_extrapolation[["sigma","function", "rmse_nn_extrap.", "rmse_mdl_interp.","n", "r"]]
    errors_extrapolation=pd.melt(errors_extrapolation,id_vars= ["sigma","function", "n", "r"], var_name="error_extrap.", 
                                 value_name= "value_extrap.")

    #Separate tanh and leaky ReLU
    errors_t_tr=errors_interpolation[errors_interpolation['function']=='tanh']
    errors_t_tt=errors_extrapolation[errors_extrapolation['function']=='tanh']

    errors_l_tr=errors_interpolation[errors_interpolation['function']=='leaky_ReLU']
    errors_l_tt=errors_extrapolation[errors_extrapolation['function']=='leaky_ReLU']

    return errors_t_tr, errors_t_tt, errors_l_tr, errors_l_tt


In [11]:

resolutions=['1x', '2x', '4e-3x']

# Initialize combined DataFrames
combined_tanh_inter = pd.DataFrame()
combined_tanh_extra = pd.DataFrame()
combined_leaky_inter = pd.DataFrame()
combined_leaky_extra = pd.DataFrame()

# Load and combine data for each resolution
for resolution in resolutions:
    errors_tanh_inter, errors_tanh_extra, errors_leaky_inter, errors_leaky_extra = get_errors(resolution)
    
    # Add resolution identifier
    errors_tanh_inter['resolution'] = resolution
    errors_tanh_extra['resolution'] = resolution
    errors_leaky_inter['resolution'] = resolution
    errors_leaky_extra['resolution'] = resolution
    
    # Combine DataFrames
    combined_tanh_inter = pd.concat([combined_tanh_inter, errors_tanh_inter])
    combined_tanh_extra = pd.concat([combined_tanh_extra, errors_tanh_extra])
    combined_leaky_inter = pd.concat([combined_leaky_inter, errors_leaky_inter])
    combined_leaky_extra = pd.concat([combined_leaky_extra, errors_leaky_extra])

,sigma,function,mae_nn_interp.,mae_nn_extrap.,mae_mdl_interp.,mae_mdl_extrap.,rmse_nn_interp.,rmse_nn_extrap.,rmse_mdl_interp.,rmse_mdl_extrap.,n,r
0,0.0,tanh,0.009377,0.057016,0.001137,0.009502,0.012386,0.060470,0.001340,0.011269,0,0
1,0.0,tanh,0.010359,0.025883,0.002827,0.312979,0.012122,0.026388,0.003454,0.457328,1,0
2,0.0,tanh,0.004791,0.083150,0.000997,0.057403,0.006643,0.086411,0.001222,0.070428,2,0
3,0.0,tanh,0.001278,0.072740,0.000314,0.042373,0.001461,0.089827,0.000406,0.049103,3,0
4,0.0,tanh,0.002132,0.012941,0.000165,0.002867,0.002459,0.014179,0.000248,0.003945,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...
655,0.2,leaky_ReLU,0.053171,0.151753,0.008551,0.143662,0.087048,0.182057,0.010784,0.165832,5,2
656,0.2,leaky_ReLU,0.101135,0.539935,0.124834,0.230996,0.127071,0.614474,0.142086,0.234844,6,2
657,0.2,leaky_ReLU,0.089813,4.261954,0.035055,0.042310,0.105589,4.445872,0.041496,0.046527,7,2
658,0.2,leaky_ReLU,0.119849,2.984129,0.102439,0.163645,0.141243,3.298494,0.114628,0.177452,8,2


,sigma,function,mae_nn_interp.,mae_nn_extrap.,mae_mdl_interp.,mae_mdl_extrap.,rmse_nn_interp.,rmse_nn_extrap.,rmse_mdl_interp.,rmse_mdl_extrap.,n,r
0,0.0,tanh,0.007060,0.022507,0.000339,0.019240,0.008476,0.023713,0.000413,0.024797,0,0
1,0.0,tanh,0.007286,0.016582,0.003138,0.072990,0.008534,0.016603,0.003706,0.083332,1,0
2,0.0,tanh,0.003471,0.074353,0.001837,291.755073,0.005279,0.077487,0.002167,515.592989,2,0
3,0.0,tanh,0.000919,0.063557,0.000155,0.057981,0.001121,0.080038,0.000226,0.092259,3,0
4,0.0,tanh,0.000879,0.012288,0.000135,0.008865,0.001065,0.013306,0.000174,0.011192,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...
655,0.2,leaky_ReLU,0.051812,1.683726,0.008533,0.143752,0.067508,1.886790,0.010756,0.165921,5,2
656,0.2,leaky_ReLU,0.073837,1.915142,0.118479,0.207158,0.091157,1.936270,0.140270,0.211444,6,2
657,0.2,leaky_ReLU,0.057993,0.819118,0.033130,0.038616,0.091006,0.881278,0.039472,0.043207,7,2
658,0.2,leaky_ReLU,0.083997,0.682737,0.056443,0.006926,0.119099,0.738111,0.070043,0.007776,8,2


,sigma,function,mae_nn_interp.,mae_nn_extrap.,mae_mdl_interp.,mae_mdl_extrap.,rmse_nn_interp.,rmse_nn_extrap.,rmse_mdl_interp.,rmse_mdl_extrap.,n,r
0,0.0,tanh,0.000853,0.013224,0.000273,0.019884,0.001288,0.013445,0.000354,0.025308,0,0
1,0.0,tanh,0.001841,0.010059,0.001976,0.116273,0.002560,0.011410,0.002339,0.152121,1,0
2,0.0,tanh,0.000912,0.050137,0.001519,0.298283,0.001101,0.069633,0.001971,0.335729,2,0
3,0.0,tanh,0.000196,0.023162,0.000119,0.174840,0.000230,0.030903,0.000175,2.283758,3,0
4,0.0,tanh,0.000404,0.017901,0.000060,0.010950,0.000509,0.020898,0.000069,0.013071,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...
655,0.2,leaky_ReLU,0.022261,0.118110,0.009094,0.151337,0.031651,0.138461,0.010507,0.172584,5,2
656,0.2,leaky_ReLU,0.015896,0.099963,0.013044,0.079376,0.024994,0.107615,0.017945,0.089668,6,2
657,0.2,leaky_ReLU,0.038772,0.126559,0.031943,0.141442,0.048350,0.150152,0.041721,0.142647,7,2
658,0.2,leaky_ReLU,0.039122,0.176571,0.030463,0.078499,0.050074,0.189842,0.036354,0.085304,8,2


In [ ]:
#Plot results
#Define figure size in cm                                                                           
cm = 1/2.54 #convert inch to cm                                                                     
width = 16*cm; height=16*cm

Extensions=['.png', '.pdf']

#colors from seaborn's colorblind palette
color_data=sns.color_palette("colorblind")[0]
color_noise=sns.color_palette("colorblind")[7]
color_ann=sns.color_palette("colorblind")[4]
color_bms=sns.color_palette("colorblind")[2]

colors={'4e-3x':['#fee0d2', '#deebf7'],
       '2x':['#fc9272', '#9ecae1'],
       '1x':['#de2d26', '#3182bd'] }

#Fonts and sizes                                                                                    
size_axis=12;size_ticks=10;size_title=5
line_w=1;marker_s=3 #width and marker size                                                          

markers = {'rmse_nn_interp.': 'o', 'rmse_mdl_interp.': 's'}; m_size=6

output_path='../../results/'

resamples=100000; bs_seed=1111
ymin=1e-4;ymax=1.15

In [ ]:
#Plot tanh interpolations

#tanh - rmse interpolation
fig=figure(figsize=(width,height), dpi=300)

for resolution in resolutions:
    data = combined_tanh_inter[combined_tanh_inter['resolution'] == resolution]
    display(data)
    sns.lineplot(
        data=data, x='sigma', y='value_interp.', hue='error_interp.', style='error_interp.',
        estimator='median', err_style="bars", errorbar=('ci', 95), n_boot=resamples, seed=bs_seed,
        markers=[markers['rmse_nn_interp.'], markers['rmse_mdl_interp.']], dashes=False, markersize=m_size, legend=True, 
        palette=colors[resolution])

#labels, limits, and ticks
plt.xlabel(r'$\sigma$',fontsize=size_axis);plt.ylabel('rmse interpolation',fontsize=size_axis)
plt.yscale("log")
xtick_labels=np.arange(0.0,0.22,0.02)
plt.xticks(xtick_labels, fontsize=size_ticks);plt.yticks(fontsize=size_ticks)
plt.minorticks_off()

plt.ylim(ymin,ymax)
sns.despine()
plt.tight_layout()

name_fig='all_interpolations_tanh' 
plt.savefig(output_path+name_fig + '.svg',dpi=300)
plt.savefig(output_path+name_fig + '.png',dpi=300)

#------------------------------------------------------------------------------------------------------------------------

In [ ]:
#Plot tanh extrapolations

#tanh - rmse extrapolation
fig=figure(figsize=(width,height), dpi=300)

for resolution in resolutions:
    data = combined_tanh_extra[combined_tanh_extra['resolution'] == resolution]
    display(data)
    sns.lineplot(
        data=data, x='sigma', y='value_extrap.', hue='error_extrap.', style='error_extrap.',
        estimator='median', err_style="bars", errorbar=('ci', 95), n_boot=resamples, seed=bs_seed,
        markers=[markers['rmse_nn_interp.'], markers['rmse_mdl_interp.']], dashes=False, markersize=m_size, legend=False, 
        palette=colors[resolution])

#labels, limits, and ticks
plt.xlabel(r'$\sigma$',fontsize=size_axis);plt.ylabel('rmse extrapolation',fontsize=size_axis)
plt.yscale("log")
xtick_labels=np.arange(0.0,0.22,0.02)
plt.xticks(xtick_labels, fontsize=size_ticks);plt.yticks(fontsize=size_ticks)
plt.minorticks_off()

plt.ylim(ymin,ymax)
sns.despine()
plt.tight_layout()

name_fig='all_extrapolations_tanh' 
plt.savefig(output_path+name_fig + '.svg',dpi=300)
plt.savefig(output_path+name_fig + '.png',dpi=300)

#------------------------------------------------------------------------------------------------------------------------

In [ ]:
#Plot leaky interpolations
fig=figure(figsize=(width,height), dpi=300)

for resolution in resolutions:
    data = combined_leaky_inter[combined_leaky_inter['resolution'] == resolution]
    sns.lineplot(
        data=data, x='sigma', y='value_interp.', hue='error_interp.', style='error_interp.',
        estimator='median', err_style="bars", errorbar=('ci', 95), n_boot=resamples, seed=bs_seed,
        markers=[markers['rmse_nn_interp.'], markers['rmse_mdl_interp.']], dashes=False, markersize=m_size, legend=False, 
        palette=colors[resolution])

#labels, limits, and ticks
plt.xlabel(r'$\sigma$',fontsize=size_axis);plt.ylabel('rmse interpolation',fontsize=size_axis)
plt.yscale("log")
xtick_labels=np.arange(0.0,0.22,0.02)
plt.xticks(xtick_labels, fontsize=size_ticks);plt.yticks(fontsize=size_ticks)
plt.minorticks_off()

plt.ylim(ymin,ymax)
sns.despine()
plt.tight_layout()

name_fig='all_interpolations_leaky' 
plt.savefig(output_path+name_fig + '.svg',dpi=300)
plt.savefig(output_path+name_fig + '.png',dpi=300)

#------------------------------------------------------------------------------------------------------------------------

In [ ]:
#Plot leaky extrapolations
fig=figure(figsize=(width,height), dpi=300)

for resolution in resolutions:
    data = combined_leaky_extra[combined_leaky_extra['resolution'] == resolution]
    sns.lineplot(
        data=data, x='sigma', y='value_extrap.', hue='error_extrap.', style='error_extrap.',
        estimator='median', err_style="bars", errorbar=('ci', 95), n_boot=resamples, seed=bs_seed,
        markers=[markers['rmse_nn_interp.'], markers['rmse_mdl_interp.']], dashes=False, markersize=m_size, legend=False, 
        palette=colors[resolution])

#labels, limits, and ticks
plt.xlabel(r'$\sigma$',fontsize=size_axis);plt.ylabel('rmse extrapolation',fontsize=size_axis)
plt.yscale("log")
xtick_labels=np.arange(0.0,0.22,0.02)
plt.xticks(xtick_labels, fontsize=size_ticks);plt.yticks(fontsize=size_ticks)
plt.minorticks_off()

plt.ylim(ymin,ymax)
sns.despine()
plt.tight_layout()

name_fig='all_extrapolationss_leaky' 
plt.savefig(output_path+name_fig + '.svg',dpi=300)
plt.savefig(output_path+name_fig + '.png',dpi=300)

#------------------------------------------------------------------------------------------------------------------------